# 05. Cihaz Yönetimi (CPU & GPU / MPS)

Bu notebook'ta donanım ivmelendirmesini (NVIDIA CUDA veya Apple Silicon MPS) kontrol etmeyi, cihaz bağımsız (agnostic) kod yazmayı, tensörleri GPU'ya taşımayı ve GPU tensörlerini NumPy formatına çevirme kurallarını öğreniyoruz.

---
### 📌 İçindekiler
1. [Donanım Kontrolü ve Agnostic Device Seçimi](#1-donanım-kontrolü-ve-agnostic-device-seçimi)
2. [Tensörleri GPU / MPS Üzerine Taşımak](#2-tensörleri-gpu--mps-üzerine-taşımak)
3. [GPU Tensörlerini NumPy'a Çevirme Hatası ve Çözümü](#3-gpu-tensörlerini-numpya-Çevirme-hatası-ve-Çözümü)
4. [PyTorch 2.x Bağlam Yöneticisi ve Varsayılan Cihaz](#4-pytorch-2x-bağlam-yöneticisi-ve-varsayılan-cihaz)


## 1. Donanım Kontrolü ve Agnostic Device Seçimi


In [1]:
import torch

In [3]:
torch.cuda.is_available() # Bilgisayarınızda GPU'nun kullanılabilir olup olmadığını kontrol edin

False

In [4]:
torch.backends.mps.is_available() # Makinenizde (Mac işlemcilerde) GPU'nun mevcut olup olmadığını kontrol edin

True

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu" # other than mac

In [6]:
device

'cpu'

In [7]:
device = "mps" if torch.backends.mps.is_available() else "cpu" # mac

In [8]:
device

'mps'

In [9]:
# veya tümü için bu kodu kullanın


In [10]:
if torch.cuda.is_available():
    device = "cuda" # Use NVIDIA GPU 
elif torch.backends.mps.is_available():
    device = "mps" # Use Apple Silicon GPU 
else:
    device = "cpu" # Default to CPU if no GPU is available

In [11]:
device

'mps'

## 2. Tensörleri GPU / MPS Üzerine Taşımak

> 💡 **Cihaz Yönetimi (.to):**
- PyTorch'ta varsayılan olarak yeni tensörler sistem **CPU** belleğinde oluşturulur.
- Modelleri ve tensörleri GPU ivmelendirmesinden faydalandırmak için `.to(device)` ile GPU'ya göndeririz.


In [12]:
tensor = torch.tensor([1, 2, 3])


In [13]:
tensor

tensor([1, 2, 3])

In [14]:
tensor.device

device(type='cpu')

In [15]:
# Kullanılabilir bir GPU'm var ancak oluşturduğum tensörler varsayılan olarak CPU'ya atanıyor.
# Ne yapabilirim?

In [16]:
# 1) Manually handle everything

In [17]:
tensor_on_gpu = tensor.to(device)

In [19]:
tensor_on_gpu

tensor([1, 2, 3], device='mps:0')

In [18]:
tensor_on_gpu.device

device(type='mps', index=0)

In [20]:
# Bu harika ama ileri geri geçiş yapmak istediğinizde size biraz sorun çıkarabilir.
# Bu arada neden tekrar CPU'ya geçmek istiyorsunuz ki?

## 3. GPU Tensörlerini NumPy'a Çevirme Hatası ve Çözümü

> ⚠️ **KRİTİK UYARI:**
- GPU veya MPS belleğinde bulunan bir tensör üzerinde doğrudan `.numpy()` çağrılamaz!
- `TypeError: can't convert mps:0 device type tensor to numpy` hatası verir.
- **Nedeni:** NumPy kütüphanesi GPU belleğini okuyamaz, yalnızca CPU (system RAM) belleğindeki verilerle çalışır.
- **Çözüm:** Tensörü önce `.cpu()` ile CPU belleğine çekip ardından `.numpy()` demek gerekir:
`tensor_on_gpu.cpu().numpy()`


In [21]:
tensor_on_gpu.numpy()

TypeError: can't convert mps:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [22]:
# Eğer GPU kullanıyorsanız bu size hata verecektir çünkü NumPy GPU'dan faydalanmaz.

In [23]:
tensor_numpy = tensor_on_gpu.cpu().numpy()

In [24]:
tensor_numpy

array([1, 2, 3])

## 4. PyTorch 2.x Bağlam Yöneticisi ve Varsayılan Cihaz

> 💡 **PyTorch 2.0+ Yenilikleri:**
- `with torch.device(device):` bloğu içerisinde oluşturulan tüm tensörler ve `nn.Linear` katmanları otomatik olarak belirlenen cihaz üzerinde oluşturulur.
- `torch.set_default_device(device)` ile notebook genelinde varsayılan cihaz belirlenebilir.


In [25]:
#2) İkinci seçenek, 2.x sürümlerinden sonra gelen bağlam yöneticisini kullanmak olacaktır.

In [26]:
device

'mps'

## 4. PyTorch 2.x Bağlam Yöneticisi ve Varsayılan Cihaz

> 💡 **PyTorch 2.0+ Yenilikleri:**
- `with torch.device(device):` bloğu içerisinde oluşturulan tüm tensörler ve `nn.Linear` katmanları otomatik olarak belirlenen cihaz üzerinde oluşturulur.
- `torch.set_default_device(device)` ile notebook genelinde varsayılan cihaz belirlenebilir.


In [27]:
with torch.device(device):
# Bu blokta oluşturulan tüm tensörler cihazda bulunacaktır.
    tensor2 = torch.tensor([1,2,3])
    layer = torch.nn.Linear(20, 30) # Henüz katmanlara değinmedik ama bu da GPU üzerinde oluşturulacak.

In [28]:
tensor2.device

device(type='mps', index=0)

In [29]:
layer.weight.device

device(type='mps', index=0)

In [32]:
tensor3 = torch.tensor([4,5,6])

In [33]:
tensor3.device

device(type='cpu')

In [34]:
#3) Üçüncü seçenek, 2.x sürümlerinden sonra gelen PyTorch için cihazın global tanımını kullanmak olacaktır.

In [35]:
device

'mps'

In [36]:
torch.set_default_device(device)

In [37]:
tensor4 = torch.tensor([1,2,3])
layer2 = torch.nn.Linear(20, 30)

In [38]:
tensor4.device

device(type='mps', index=0)

In [39]:
layer2.weight.device

device(type='mps', index=0)

In [40]:
tensor3.device

device(type='cpu')

---
### 📌 05. Cihaz Yönetimi - Özet Cheat Sheet

| İşlem | Kod | Açıklama |
| :--- | :--- | :--- |
| **CUDA İvmelendirme** | `torch.cuda.is_available()` | NVIDIA GPU var mı? |
| **Apple Silicon İvmelendirme** | `torch.backends.mps.is_available()` | Mac M1/M2/M3/M4 GPU var mı? |
| **GPU'ya Taşıma** | `tensor.to(device)` | Tensörü belirlenen GPU cihazına gönderir |
| **NumPy Dönüşümü** | `tensor.cpu().numpy()` | ⚠️ GPU tensörünü önce CPU'ya çekmek ŞARTTIR! |
| **PyTorch 2.x Default** | `torch.set_default_device(device)` | Tüm yeni tensörleri otomatik GPU'da oluşturur |
